In [1]:
!pip install scikit-learn tqdm

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.2 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 2.1/11.2 MB 8.4 MB/s eta 0:00:02
   ------------------- -------------------- 5.5/11.2 MB 10.5 MB/s eta 0:00:01
   ----------------------------- ---------- 8.1/11.2 MB 10.7 MB/s eta 0:00:01
   ------------------------------------- -- 10.5/11.2 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 10.4 MB/s  0:00:01

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-lea


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import shutil
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm

# =========================
# CONFIG
# =========================
image_dir = "images"
label_dir = "labels"

output_dir = "dataset_split"

train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# =========================
# CREATE OUTPUT STRUCTURE
# =========================
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

# =========================
# LOAD FILES
# =========================
image_files = sorted(os.listdir(image_dir))

data = []
labels_list = []

for img_file in image_files:
    label_file = os.path.splitext(img_file)[0] + ".txt"
    label_path = os.path.join(label_dir, label_file)

    if not os.path.exists(label_path):
        continue

    # Read class (first object in file)
    with open(label_path, "r") as f:
        first_line = f.readline().strip()
        if first_line == "":
            continue
        class_id = int(first_line.split()[0])

    data.append((img_file, label_file))
    labels_list.append(class_id)

print(f"Total valid samples: {len(data)}")

# =========================
# STRATIFIED SPLIT
# =========================
train_data, temp_data, train_labels, temp_labels = train_test_split(
    data, labels_list,
    test_size=(1 - train_ratio),
    stratify=labels_list,
    random_state=42
)

val_size = val_ratio / (val_ratio + test_ratio)

val_data, test_data, _, _ = train_test_split(
    temp_data, temp_labels,
    test_size=(1 - val_size),
    stratify=temp_labels,
    random_state=42
)

# =========================
# COPY FILES
# =========================
def copy_files(dataset, split_name):
    for img_file, label_file in tqdm(dataset, desc=f"Copying {split_name}"):
        shutil.copy(
            os.path.join(image_dir, img_file),
            os.path.join(output_dir, "images", split_name, img_file)
        )
        
        shutil.copy(
            os.path.join(label_dir, label_file),
            os.path.join(output_dir, "labels", split_name, label_file)
        )

copy_files(train_data, "train")
copy_files(val_data, "val")
copy_files(test_data, "test")

# =========================
# PRINT STATS
# =========================
print("\nDataset Split Summary:")
print(f"Train: {len(train_data)} images")
print(f"Validation: {len(val_data)} images")
print(f"Test: {len(test_data)} images")

# Optional: class distribution
def get_class_distribution(dataset):
    counter = Counter()
    for _, label_file in dataset:
        with open(os.path.join(label_dir, label_file)) as f:
            cls = int(f.readline().split()[0])
            counter[cls] += 1
    return counter

print("\nClass Distribution:")
print("Train:", get_class_distribution(train_data))
print("Val:", get_class_distribution(val_data))
print("Test:", get_class_distribution(test_data))

Total valid samples: 2954


Copying test: 100%|██████████████████████████████████████████████████████████████████| 296/296 [00:05<00:00, 51.05it/s]



Dataset Split Summary:
Train: 2067 images
Validation: 591 images
Test: 296 images

Class Distribution:
Train: Counter({0: 754, 1: 753, 2: 560})
Val: Counter({0: 215, 1: 215, 2: 161})
Test: Counter({0: 108, 1: 108, 2: 80})
